# Phase 1: Deploying Llama-3-8B on Google Colab (T4 GPU)
### Mentorship Alignment: Md Shoaib Ahmed (Trustworthy AI Lab, Boise State University)

> 📖 **Companion Guide & Documentation**: [notes/phase_1/phase_1_colab_guide.md](../notes/phase_1/phase_1_colab_guide.md)

---

## 📋 Step-by-Step Prerequisites Checklist

Before executing the cells below, complete these 4 simple setup steps:

### 1. Create a Free Hugging Face Account
- Navigate to: https://huggingface.co/join
- Register and verify your email.

### 2. Accept Meta's Community License for Llama-3-8B
- Go to: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
- Fill in First Name, Last Name, Organization (enter "Student" or "Independent Researcher"), and Country.
- Check "I accept the Meta Llama 3 Community License Agreement" and click **Submit**.
- Access is granted immediately within seconds.

### 3. Generate a Hugging Face Access Token
- Go to: https://huggingface.co/settings/tokens
- Click **Create new token**.
- Select Token Type: **Read**.
- Name: `colab-llama3`
- Click **Create token** and copy the resulting `hf_...` token.

### 4. Enable Google Colab T4 GPU
- In the Colab menu bar above, go to: **Runtime -> Change runtime type**.
- Under **Hardware accelerator**, select **T4 GPU**.
- Click **Save** and verify connection in the top right corner.

## Cell 1: Hardware Verification
Verify that Google Colab has attached an NVIDIA Tesla T4 GPU (~15 GB VRAM).

In [ ]:
!nvidia-smi

## Cell 2: Install Core Dependencies
Install PyTorch, Hugging Face Transformers, BitsAndBytes (4-bit quantization kernel), Accelerate, and Hugging Face Hub.

In [ ]:
!pip install -q -U torch transformers bitsandbytes accelerate huggingface_hub

## Cell 3: Hugging Face Authentication
Authenticate to pull gated weights from Meta's repository using your Read token.

In [ ]:
from huggingface_hub import login

# Enter your HF read token when prompted
login()

## Cell 4: Load Model with 4-bit NF4 Quantization
Configure BitsAndBytesConfig with NormalFloat4 (NF4), double quantization, and float16 compute dtype.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Define 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. Load model with auto device placement onto T4 GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully onto GPU!")

## Cell 5: Programmatic VRAM Memory Audit
Inspect actual VRAM allocated to verify the ~5.5 GB footprint.

In [ ]:
allocated_vram = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_vram = torch.cuda.memory_reserved() / (1024 ** 3)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"Allocated VRAM : {allocated_vram:.2f} GB")
print(f"Reserved VRAM  : {reserved_vram:.2f} GB")
print(f"Total GPU VRAM : {total_vram:.2f} GB")
print(f"Free Headroom  : {(total_vram - reserved_vram):.2f} GB")

## Cell 6: Modular Generation Function
Define a modular inference function adhering strictly to Llama-3's official chat template.

In [ ]:
def generate_response(prompt: str, system_prompt: str = "You are a helpful and precise research assistant.", max_new_tokens: int = 150, temperature: float = 0.6):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True if temperature > 0 else False,
        pad_token_id=tokenizer.eos_token_id
    )
    
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## Cell 7: First Smoke Test Generation
Execute baseline test prompt and verify post-inference VRAM stability.

In [ ]:
test_prompt = "Explain in one sentence why social media comments influence people's belief in news headlines."
response = generate_response(test_prompt)

print("Prompt:", test_prompt)
print("\nResponse:", response)

vram_after = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"\nVRAM After Inference: {vram_after:.2f} GB")